
# Capstone 2 — Feature Engineering & Preprocessing (Continuation from EDA Output)

This notebook **starts from the dataset produced by EDA**:
- `data/processed/insurance_final_for_modeling.csv`

It completes the Springboard **Pre-processing and Training Data Development** step:
1) Dummy/indicator features (already created in EDA; validated here)  
2) Standardize numeric feature magnitude (StandardScaler)  
3) Train/test split (train_test_split)  
4) Save final artifacts for modeling


## 1. Load EDA-Generated Dataset

In [3]:

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

EDA_OUT_PATH = "data/processed/insurance_final_for_modeling.csv"

if os.path.exists(EDA_OUT_PATH):
    df_model = pd.read_csv(EDA_OUT_PATH)
    print(f"Loaded EDA output: {EDA_OUT_PATH}")
else:
    # Fallback: if the file isn't present, recreate it from insurance_cleaned.csv
    # so this notebook remains runnable end-to-end.
    print(f"EDA output not found at {EDA_OUT_PATH}. Recreating from insurance_cleaned.csv...")
    df = pd.read_csv("insurance_cleaned.csv")
    df_model = pd.get_dummies(df, drop_first=True)
    os.makedirs("data/processed", exist_ok=True)
    df_model.to_csv(EDA_OUT_PATH, index=False)
    print(f"Recreated and saved: {EDA_OUT_PATH}")

df_model.head()


Loaded EDA output: data/processed/insurance_final_for_modeling.csv


,age,sex,bmi,children,smoker,region,charges,insuranceclaim
0,19,0,27.900,0,1,3,16884.92400,1
1,18,1,33.770,1,0,2,1725.55230,1
2,28,1,33.000,3,0,2,4449.46200,0
3,33,1,22.705,0,0,1,21984.47061,0
4,32,1,28.880,0,0,1,3866.85520,1


In [4]:
df_model.shape

(1337, 8)


## 2. Validate Dummy Features Are Present

EDA should have already one-hot encoded categorical variables.  
We confirm that:
- there are **no object (string) columns** remaining
- all features are numeric


In [5]:

object_cols = df_model.select_dtypes(include=["object"]).columns.tolist()
object_cols


[]

In [7]:

if len(object_cols) == 0:
    print("No object columns found. Dummy features already created in EDA.")
else:
    print("Object columns still exist. Creating dummies now...")
    df_model = pd.get_dummies(df_model, columns=object_cols, drop_first=True)
    print("Dummies created. New shape:", df_model.shape)


No object columns found. Dummy features already created in EDA.



## 3. Define Targets and Feature Matrix

This dataset supports two modeling targets:
- **Regression:** `charges`
- **Classification:** `insuranceclaim`


In [8]:

TARGET_REG = "charges"
TARGET_CLF = "insuranceclaim"

missing_targets = [t for t in [TARGET_REG, TARGET_CLF] if t not in df_model.columns]
missing_targets


[]

In [9]:

if missing_targets:
    raise ValueError(f"Missing expected target columns: {missing_targets}. Check your dataset columns.")
    
X = df_model.drop(columns=[TARGET_REG, TARGET_CLF])
y_reg = df_model[TARGET_REG]
y_clf = df_model[TARGET_CLF]

X.shape, y_reg.shape, y_clf.shape


((1337, 6), (1337,), (1337,))


## 4. Standardize Numeric Features (StandardScaler)

Even after dummy encoding, scaling can help many models.
We scale **all feature columns in X** (targets are not scaled).


In [10]:

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

X_scaled.head()


,age,sex,bmi,children,smoker,region
0,-1.440418,-1.009771,-0.453160,-0.909234,1.969660,1.343163
1,-1.511647,0.990324,0.509422,-0.079442,-0.507702,0.438017
2,-0.799350,0.990324,0.383155,1.580143,-0.507702,0.438017
3,-0.443201,0.990324,-1.305052,-0.909234,-0.507702,-0.467128
4,-0.514431,0.990324,-0.292456,-0.909234,-0.507702,-0.467128



## 5. Train/Test Split

We create train/test splits for both regression and classification targets.
(We split once for X, then align both targets to the same split indices.)


In [11]:

X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X_scaled, y_reg, test_size=0.2, random_state=42
)

# Use the same indices to split classification target for consistency
y_clf_train = y_clf.loc[y_reg_train.index]
y_clf_test  = y_clf.loc[y_reg_test.index]

X_train.shape, X_test.shape, y_reg_train.shape, y_clf_train.shape


((1069, 6), (268, 6), (1069,), (1069,))


## 6. Save Final Modeling Artifacts

These files can be loaded directly in the Modeling notebook.


In [13]:

import os

os.makedirs("data/processed", exist_ok=True)

X_train.to_csv("data/processed/X_train_scaled.csv", index=False)
X_test.to_csv("data/processed/X_test_scaled.csv", index=False)

y_reg_train.to_csv("data/processed/y_reg_train.csv", index=False)
y_reg_test.to_csv("data/processed/y_reg_test.csv", index=False)

y_clf_train.to_csv("data/processed/y_clf_train.csv", index=False)
y_clf_test.to_csv("data/processed/y_clf_test.csv", index=False)

print(" Saved:")
print("- data/processed/X_train_scaled.csv")
print("- data/processed/X_test_scaled.csv")
print("- data/processed/y_reg_train.csv, y_reg_test.csv")
print("- data/processed/y_clf_train.csv, y_clf_test.csv")


 Saved:
- data/processed/X_train_scaled.csv
- data/processed/X_test_scaled.csv
- data/processed/y_reg_train.csv, y_reg_test.csv
- data/processed/y_clf_train.csv, y_clf_test.csv



## 7. Summary

- Loaded EDA-generated dataset (`insurance_final_for_modeling.csv`)
- Confirmed dummy features (no object columns)
- Standardized feature magnitudes with StandardScaler
- Split into training/testing sets
- Saved final modeling-ready artifacts
